# Experimento Gen/01 — Caracterização dos Materiais 2D UWBG

**Objetivo:** identificar padrões físico-químicos e estruturais que distinguem materiais 2D
com Ultra-Wide Bandgap (UWBG, gap_HSE > 3.4 eV) dos demais materiais do C2DB.
Os filtros obtidos aqui alimentam o Experimento Gen/02 como função de score `S_chem(C)`.

## Papel neste trabalho

```
Triagem (Exps 01–12)  →  [Este notebook]  →  Geração guiada (Gen/02)
                         Caracterização UWBG
                         Filtros químicos
                         Classificador S_chem
```

## Estrutura

1. Carregamento dos dados (C2DB + candidatos run 005)
2. Panorama: UWBG vs Não-UWBG no C2DB
3. Extração de features físico-químicas (pymatgen)
4. Frequência de elementos — UWBG vs Não-UWBG
5. Distribuições: eletronegatividade, espessura, formação, ehull
6. Análise dos candidatos run 005 — TP vs FP vs Novel
7. Classificador interpretável (Random Forest + Regressão Logística)
8. Importância das features
9. Filtros químicos — exportação para Gen/02
10. Resumo executivo

**Dados de entrada:**
- `data/raw/c2db.db` — base C2DB completa (16.905 materiais 2D)
- `runs/001_megnet_finetune_c2db/outputs/uwbg_candidates.csv` — candidatos UWBG do melhor modelo

**Saídas:**
- `outputs/c2db_hse_features.csv` — features de todos os materiais com HSE calculado
- `outputs/candidates_classified.csv` — candidatos com class TP/FP/novel + features
- `outputs/chemical_filters.json` — filtros exportáveis para Gen/02
- `outputs/rf_classifier.joblib` — Random Forest treinado (S_chem)
- `figures/` — todos os gráficos


In [ ]:
from __future__ import annotations
import warnings
warnings.filterwarnings('ignore')
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import ase.db
from pymatgen.core import Composition, Element
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score,
    ConfusionMatrixDisplay, confusion_matrix,
)
import joblib

import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))

from common import DATA_DIR, FINAL_ROOT, REPO_ROOT, RUNS_DIR, ensure_run_dir, required_path

# ── Caminhos ──────────────────────────────────────────────────────────────────
RUN_DIR = ensure_run_dir('006', 'uwbg_characterization')
C2DB_PATH = required_path(DATA_DIR / 'raw' / 'c2db.db', 'C2DB')
CANDIDATES_PATH = required_path(RUNS_DIR / '001_megnet_finetune_c2db' / 'outputs' / 'uwbg_candidates.csv', 'candidatos UWBG')
OUT_DIR = RUN_DIR / 'outputs'
FIG_DIR = RUN_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Constantes ────────────────────────────────────────────────────────────────
UWBG_THR = 3.4   # eV — limiar de classificação UWBG

# ── Estilo dos gráficos ───────────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.titlesize': 12, 'axes.labelsize': 11})
C_UWBG   = '#e76f51'
C_NUWBG  = '#2a9d8f'
C_NOVEL  = '#8338ec'
C_FP     = '#f4a261'
print('Ambiente pronto.')


## 1. Carregamento dos Dados

### 1.1 C2DB completo
Carregamos todos os 16.905 materiais e extraímos os campos relevantes.
Mantemos apenas os 3.363 que têm `gap_hse` calculado para análise supervisionada.


In [ ]:
db = ase.db.connect(C2DB_PATH)

records = []
for row in db.select():
    kvp  = row.key_value_pairs
    cell = row.cell
    a = np.linalg.norm(cell[0])
    b = np.linalg.norm(cell[1])
    records.append({
        'uid'        : kvp.get('uid', ''),
        'formula'    : row.formula,
        'natoms'     : row.natoms,
        'gap_pbe'    : kvp.get('gap', np.nan),
        'gap_hse'    : kvp.get('gap_hse', np.nan),
        'dyn_stab'   : kvp.get('dyn_stab', 'Unknown'),
        'ehull'      : kvp.get('ehull', np.nan),
        'hform'      : kvp.get('hform', np.nan),
        'thickness'  : kvp.get('thickness', np.nan),
        'layergroup' : kvp.get('layergroup', ''),
        'lgnum'      : kvp.get('lgnum', np.nan),
        'bravais_type': kvp.get('bravais_type', ''),
        'is_magnetic': int(bool(kvp.get('is_magnetic', False))),
        'a_lat'      : a,
        'b_lat'      : b,
    })

df_c2db = pd.DataFrame(records)

# Subconjunto com HSE disponível (base para análise supervisionada)
df_hse = df_c2db[df_c2db['gap_hse'].notna() & (df_c2db['gap_hse'] > 0)].copy()
df_hse['is_uwbg'] = (df_hse['gap_hse'] > UWBG_THR).astype(int)

print(f'C2DB total            : {len(df_c2db):,}')
print(f'Com gap_hse calculado : {len(df_hse):,}')
print(f'  → UWBG (>{UWBG_THR} eV)     : {df_hse["is_uwbg"].sum():,} ({df_hse["is_uwbg"].mean()*100:.1f}%)')
print(f'  → Não-UWBG           : {(1-df_hse["is_uwbg"]).sum():,}')
print(f'  → dyn_stab=Yes       : {(df_hse["dyn_stab"]=="Yes").sum():,}')


### 1.2 Candidatos UWBG — Run 005

O run 005 (MEGNet fine-tune) previu 588 candidatos com `uwbg_pred=True`.
Classificamos cada um como **TP** (HSE real > 3.4 eV), **FP** (HSE real ≤ 3.4 eV)
ou **Novel** (sem HSE calculado).


In [ ]:
df_cands = pd.read_csv(CANDIDATES_PATH)
df_uwbg  = df_cands[df_cands['uwbg']].copy()   # apenas os preditos como UWBG

def classify_candidate(row):
    if pd.isna(row['gap_hse_true']):
        return 'Novel'
    return 'TP' if row['gap_hse_true'] > UWBG_THR else 'FP'

df_uwbg['class'] = df_uwbg.apply(classify_candidate, axis=1)

# Merge com C2DB para features estruturais
struct_cols = ['uid', 'dyn_stab', 'ehull', 'hform', 'thickness',
               'layergroup', 'lgnum', 'is_magnetic', 'a_lat', 'b_lat']
df_uwbg = df_uwbg.merge(df_c2db[struct_cols], on='uid', how='left')

print('=== Candidatos UWBG (run 005) ===')
print(df_uwbg['class'].value_counts().to_string())
print(f'\nTotal candidatos UWBG preditos : {len(df_uwbg)}')
precision = (df_uwbg['class'] == 'TP').sum() / (df_uwbg['class'] != 'Novel').sum()
print(f'Precisão (TP/TP+FP, validados) : {precision:.3f}')


## 2. Panorama — UWBG vs Não-UWBG no C2DB


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Distribuição gap_hse ---
ax = axes[0]
bins = np.linspace(0, 12, 60)
uwbg_mask = df_hse['is_uwbg'] == 1
ax.hist(df_hse.loc[~uwbg_mask, 'gap_hse'], bins=bins,
        color=C_NUWBG, alpha=0.75, label='Não-UWBG', density=True)
ax.hist(df_hse.loc[uwbg_mask, 'gap_hse'], bins=bins,
        color=C_UWBG, alpha=0.75, label='UWBG (>3.4 eV)', density=True)
ax.axvline(UWBG_THR, color='black', linestyle='--', linewidth=1.2,
           label=f'Limiar {UWBG_THR} eV')
ax.set_xlabel('gap_HSE (eV)')
ax.set_ylabel('Densidade')
ax.set_title('Distribuição gap_HSE — C2DB completo')
ax.legend(fontsize=9)

# --- Distribuição gap_pbe vs gap_hse (scatter) ---
ax = axes[1]
ax.scatter(df_hse.loc[~uwbg_mask, 'gap_pbe'],
           df_hse.loc[~uwbg_mask, 'gap_hse'],
           s=8, alpha=0.3, color=C_NUWBG, label='Não-UWBG')
ax.scatter(df_hse.loc[uwbg_mask, 'gap_pbe'],
           df_hse.loc[uwbg_mask, 'gap_hse'],
           s=12, alpha=0.6, color=C_UWBG, label='UWBG')
ax.axhline(UWBG_THR, color='black', linestyle='--', linewidth=0.8)
lim = max(df_hse['gap_hse'].max(), df_hse['gap_pbe'].max()) * 1.05
ax.plot([0, lim], [0, lim], 'k:', linewidth=0.7, label='PBE = HSE')
ax.set_xlabel('gap_PBE (eV)')
ax.set_ylabel('gap_HSE (eV)')
ax.set_title('PBE vs HSE — underestimação sistemática')
ax.legend(fontsize=9, markerscale=2)

plt.tight_layout()
plt.savefig(FIG_DIR / 'panorama_gap_c2db.png', bbox_inches='tight')
plt.show()


## 3. Extração de Features Físico-Químicas

Extraímos features composicionais via pymatgen para cada material com HSE calculado.
Estas mesmas features serão usadas no Gen/02 para pontuar estruturas geradas.

**Features composicionais** (derivadas da fórmula):
- Flags de presença de elementos por grupo (halogênios, alcalinos, alcalino-terrosos, metais de transição)
- Eletronegatividade: média, máxima, mínima, amplitude
- Período, grupo, raio covalente, massa atômica médios
- Número de espécies distintas

**Features estruturais** (do C2DB):
- Espessura da camada (`thickness`)
- Energia acima do hull (`ehull`)
- Entalpia de formação (`hform`)
- Estabilidade dinâmica (`dyn_stab_yes`)
- Número de átomos na cela unitária
- Flag magnético


In [ ]:
HALIDES     = frozenset(['F', 'Cl', 'Br', 'I'])
ALKALIS     = frozenset(['Li', 'Na', 'K', 'Rb', 'Cs'])
ALK_EARTHS  = frozenset(['Be', 'Mg', 'Ca', 'Sr', 'Ba'])
TM_Z        = set(range(21, 31)) | set(range(39, 49)) | set(range(72, 81))


def extract_features(formula: str) -> dict | None:
    """Retorna dict de features ou None se a fórmula for inválida."""
    try:
        comp = Composition(formula)
    except Exception:
        return {}  # retorna vazio → NaN nas colunas de feature

    elements = list(comp.elements)
    syms     = {str(e) for e in elements}
    ens      = [e.X for e in elements if e.X is not None]
    periods  = [e.row for e in elements]
    groups   = [e.group for e in elements]
    cov_r    = [float(e.atomic_radius_calculated or e.atomic_radius or 0)
                for e in elements]
    masses   = [float(e.atomic_mass) for e in elements]

    return {
        'n_species'           : len(elements),
        'has_F'               : int('F'  in syms),
        'has_Cl'              : int('Cl' in syms),
        'has_Br'              : int('Br' in syms),
        'has_I'               : int('I'  in syms),
        'has_O'               : int('O'  in syms),
        'has_N'               : int('N'  in syms),
        'has_S'               : int('S'  in syms),
        'has_halide'          : int(bool(syms & HALIDES)),
        'has_alkali'          : int(bool(syms & ALKALIS)),
        'has_alkaline_earth'  : int(bool(syms & ALK_EARTHS)),
        'has_transition_metal': int(any(e.Z in TM_Z for e in elements)),
        'mean_en'             : float(np.mean(ens))    if ens else np.nan,
        'max_en'              : float(np.max(ens))     if ens else np.nan,
        'min_en'              : float(np.min(ens))     if ens else np.nan,
        'range_en'            : float(np.ptp(ens))     if ens else np.nan,
        'mean_period'         : float(np.mean(periods)),
        'max_period'          : float(np.max(periods)),
        'mean_group'          : float(np.mean(groups)),
        'mean_cov_radius'     : float(np.mean([r for r in cov_r if r > 0]))
                                if any(r > 0 for r in cov_r) else np.nan,
        'mean_atomic_mass'    : float(np.mean(masses)),
    }


print('Extraindo features composicionais…')
feat_rows = df_hse['formula'].apply(extract_features)
df_comp   = pd.DataFrame(feat_rows.tolist(), index=df_hse.index)

# Combinar composicional + estrutural
df_feat_full = pd.concat([
    df_hse[['uid', 'formula', 'gap_hse', 'gap_pbe', 'is_uwbg',
             'ehull', 'hform', 'thickness', 'natoms',
             'is_magnetic', 'dyn_stab', 'a_lat', 'b_lat', 'lgnum']].reset_index(drop=True),
    df_comp.reset_index(drop=True),
], axis=1)

df_feat_full['dyn_stab_yes'] = (df_feat_full['dyn_stab'] == 'Yes').astype(int)
df_feat_full['ab_ratio']     = df_feat_full['a_lat'] / df_feat_full['b_lat'].replace(0, np.nan)

df_feat_full.to_csv(OUT_DIR / 'c2db_hse_features.csv', index=False)
print(f'Shape: {df_feat_full.shape}')
print(f'Nulos por coluna:\n{df_feat_full.isnull().sum()[df_feat_full.isnull().sum() > 0]}')


## 4. Frequência de Elementos — UWBG vs Não-UWBG

Calculamos em quantos materiais de cada classe cada elemento aparece.
Isso revela quais elementos são enriquecidos nos UWBG.


In [ ]:
from collections import Counter


def element_counts(formulas):
    cnt = Counter()
    for f in formulas:
        try:
            comp = Composition(f)
            cnt.update(str(e) for e in comp.elements)
        except Exception:
            pass
    return cnt


uwbg_forms  = df_feat_full.loc[df_feat_full['is_uwbg'] == 1, 'formula']
nuwbg_forms = df_feat_full.loc[df_feat_full['is_uwbg'] == 0, 'formula']

cnt_uwbg  = element_counts(uwbg_forms)
cnt_nuwbg = element_counts(nuwbg_forms)

n_uwbg  = len(uwbg_forms)
n_nuwbg = len(nuwbg_forms)

# Fração de ocorrência (% materiais que contêm o elemento)
all_elems = sorted(set(cnt_uwbg) | set(cnt_nuwbg))
df_elem = pd.DataFrame({
    'element': all_elems,
    'freq_uwbg' : [cnt_uwbg.get(e, 0) / n_uwbg  * 100 for e in all_elems],
    'freq_nuwbg': [cnt_nuwbg.get(e, 0) / n_nuwbg * 100 for e in all_elems],
})
df_elem['ratio'] = (df_elem['freq_uwbg'] + 1) / (df_elem['freq_nuwbg'] + 1)

# Top 20 elementos mais enriquecidos em UWBG (maior ratio)
top_uwbg = df_elem.nlargest(20, 'ratio')
# Top 20 mais presentes em UWBG (maior freq_uwbg)
top_freq = df_elem.nlargest(20, 'freq_uwbg')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Frequência absoluta (top 20 em UWBG) ---
ax = axes[0]
x  = np.arange(len(top_freq))
w  = 0.35
ax.bar(x - w/2, top_freq['freq_uwbg'],  w, label='UWBG',     color=C_UWBG,  alpha=0.85)
ax.bar(x + w/2, top_freq['freq_nuwbg'], w, label='Não-UWBG', color=C_NUWBG, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top_freq['element'], rotation=45, ha='right')
ax.set_ylabel('% materiais contendo o elemento')
ax.set_title('Elementos mais frequentes nos UWBG')
ax.legend()

# --- Razão de enriquecimento (freq_uwbg / freq_nuwbg) ---
ax = axes[1]
colors = [C_UWBG if r > 1 else C_NUWBG for r in top_uwbg['ratio']]
ax.barh(top_uwbg['element'], top_uwbg['ratio'], color=colors, alpha=0.85)
ax.axvline(1, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Razão de enriquecimento (UWBG / Não-UWBG)')
ax.set_title('Top 20 elementos enriquecidos nos UWBG')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(FIG_DIR / 'element_frequency.png', bbox_inches='tight')
plt.show()

print('\nTop 10 elementos mais enriquecidos em UWBG:')
print(df_elem.nlargest(10, 'ratio')[['element', 'freq_uwbg', 'freq_nuwbg', 'ratio']].to_string(index=False))


## 5. Distribuições de Features — UWBG vs Não-UWBG

Comparamos as distribuições das features contínuas mais relevantes entre as duas classes.


In [ ]:
feat_plot = [
    ('range_en',     'Amplitude de eletronegatividade\n(max - min Pauling)', False),
    ('mean_en',      'Eletronegatividade média', False),
    ('max_en',       'Eletronegatividade máxima\n(proxy ânion)', False),
    ('thickness',    'Espessura da camada (Å)', False),
    ('hform',        'Entalpia de formação (eV/átomos)', False),
    ('ehull',        'Energia acima do hull (eV/átomo)', True),
    ('mean_period',  'Período médio', False),
    ('n_species',    'Nº de espécies', False),
]

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
axes = axes.flatten()

for ax, (feat, label, clip) in zip(axes, feat_plot):
    data_uwbg  = df_feat_full.loc[df_feat_full['is_uwbg'] == 1, feat].dropna()
    data_nuwbg = df_feat_full.loc[df_feat_full['is_uwbg'] == 0, feat].dropna()
    if clip:
        data_uwbg  = data_uwbg.clip(upper=data_uwbg.quantile(0.95))
        data_nuwbg = data_nuwbg.clip(upper=data_nuwbg.quantile(0.95))
    bp = ax.boxplot([data_nuwbg, data_uwbg],
                    labels=['Não-UWBG', 'UWBG'],
                    patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], [C_NUWBG + '66', C_UWBG + '66']):
        patch.set_facecolor(color)
    ax.set_ylabel(label, fontsize=9)
    ax.set_title(label.split('\n')[0], fontsize=10)

plt.suptitle('Distribuição de features — UWBG vs Não-UWBG (C2DB com HSE)', y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_distributions.png', bbox_inches='tight')
plt.show()


## 6. Análise dos Candidatos Run 005 — TP vs FP vs Novel

Focamos nos 588 materiais preditos como UWBG pelo MEGNet fine-tune e analisamos
o que distingue os verdadeiros positivos dos falsos positivos.


In [ ]:
# Extrair features para os candidatos UWBG
feat_cands = df_uwbg['formula'].apply(extract_features)
df_cands_feat = pd.concat([
    df_uwbg[['uid', 'formula', 'gap_pbe', 'gap_hse_true', 'gap_hse_pred',
              'class', 'ehull', 'hform', 'thickness', 'is_magnetic',
              'dyn_stab', 'n_atoms']].reset_index(drop=True),
    pd.DataFrame(feat_cands.tolist()).reset_index(drop=True),
], axis=1)

# Alinhar nomes de colunas com df_feat_full para reutilizar FEAT_COLS no Gen/02
df_cands_feat = df_cands_feat.rename(columns={'n_atoms': 'natoms'})
df_cands_feat['dyn_stab_yes'] = (df_cands_feat['dyn_stab'] == 'Yes').astype(int)

df_cands_feat.to_csv(OUT_DIR / 'candidates_classified.csv', index=False)
print(df_cands_feat['class'].value_counts())
print(f'\nShape: {df_cands_feat.shape}')


In [ ]:
# Comparação TP vs FP — features selecionadas
df_tp = df_cands_feat[df_cands_feat['class'] == 'TP']
df_fp = df_cands_feat[df_cands_feat['class'] == 'FP']
df_nov = df_cands_feat[df_cands_feat['class'] == 'Novel']

feats_compare = ['range_en', 'max_en', 'mean_en', 'has_halide',
                 'has_transition_metal', 'thickness', 'n_species', 'mean_period']

summary = pd.DataFrame({
    'TP (n={:d})'.format(len(df_tp))    : df_tp[feats_compare].mean(),
    'FP (n={:d})'.format(len(df_fp))    : df_fp[feats_compare].mean(),
    'Novel (n={:d})'.format(len(df_nov)): df_nov[feats_compare].mean(),
}).round(3)

print('=== Médias por classe (candidatos preditos UWBG) ===')
print(summary.to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
classes  = ['TP', 'FP', 'Novel']
colors_c = [C_UWBG, C_FP, C_NOVEL]

for ax, (feat, label) in zip(
    axes,
    [('range_en', 'Amplitude eletronegatividade'),
     ('gap_hse_pred', 'gap_HSE predito (eV)'),
     ('has_transition_metal', 'Fração com metal de transição')]
):
    data = [df_cands_feat.loc[df_cands_feat['class'] == cls, feat].dropna()
            for cls in classes]
    if feat == 'has_transition_metal':
        means = [d.mean() for d in data]
        ax.bar(classes, means, color=colors_c, alpha=0.85)
        ax.set_ylabel('Fração')
    else:
        bp = ax.boxplot(data, labels=classes, patch_artist=True,
                        medianprops=dict(color='black', linewidth=2))
        for patch, col in zip(bp['boxes'], colors_c):
            patch.set_facecolor(col + '55')
        ax.set_ylabel(label)
    ax.set_title(label)

plt.suptitle('TP vs FP vs Novel — candidatos UWBG run 005', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'tp_fp_novel_comparison.png', bbox_inches='tight')
plt.show()


## 7. Classificador Interpretável

Treinamos dois classificadores para distinguir UWBG (gap_HSE > 3.4 eV) de não-UWBG
usando **apenas features composicionais e estruturais** (sem usar o MEGNet).

- **Regressão Logística** (L2): modelo linear interpretável
- **Random Forest** (100 árvores): modelo não-linear, mais expressivo

O Random Forest será exportado como `S_chem(C)` para o Experimento Gen/02.

**Dataset de treino:** todos os materiais C2DB com gap_HSE calculado (n ≈ 3363).
**Validação cruzada:** 5-fold estratificado.


In [ ]:
FEAT_COLS = [
    'n_species', 'has_F', 'has_Cl', 'has_Br', 'has_I', 'has_O', 'has_N', 'has_S',
    'has_halide', 'has_alkali', 'has_alkaline_earth', 'has_transition_metal',
    'mean_en', 'max_en', 'min_en', 'range_en',
    'mean_period', 'max_period', 'mean_group',
    'mean_cov_radius', 'mean_atomic_mass',
    'thickness', 'natoms', 'is_magnetic', 'dyn_stab_yes',
]

df_ml = df_feat_full[FEAT_COLS + ['is_uwbg']].dropna()
X = df_ml[FEAT_COLS].values
y = df_ml['is_uwbg'].values

print(f'Dataset classificador: {X.shape[0]} materiais | '
      f'UWBG: {y.sum()} ({y.mean()*100:.1f}%) | Não-UWBG: {(1-y).sum()}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Regressão Logística
pipe_lr = Pipeline([('scaler', StandardScaler()),
                    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))])
cv_f1_lr  = cross_val_score(pipe_lr,  X, y, cv=cv, scoring='f1')
cv_auc_lr = cross_val_score(pipe_lr,  X, y, cv=cv, scoring='roc_auc')

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                             class_weight='balanced', random_state=42, n_jobs=-1)
cv_f1_rf  = cross_val_score(rf, X, y, cv=cv, scoring='f1')
cv_auc_rf = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')

print('\n=== Validação Cruzada 5-fold ===')
print(f'Logistic Regression — F1: {cv_f1_lr.mean():.3f} ± {cv_f1_lr.std():.3f}  '
      f'AUC: {cv_auc_lr.mean():.3f} ± {cv_auc_lr.std():.3f}')
print(f'Random Forest       — F1: {cv_f1_rf.mean():.3f} ± {cv_f1_rf.std():.3f}  '
      f'AUC: {cv_auc_rf.mean():.3f} ± {cv_auc_rf.std():.3f}')

# Treinar RF final em todos os dados
rf.fit(X, y)
pipe_lr.fit(X, y)

# Relatório completo no dataset completo (indicativo — sem split)
print('\n=== Relatório RF (treino completo — apenas indicativo) ===')
print(classification_report(y, rf.predict(X), target_names=['Não-UWBG', 'UWBG']))


In [ ]:
# Matriz de confusão (5-fold, fold final)
from sklearn.model_selection import StratifiedKFold
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y))
tr, te = folds[-1]

rf_fold = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                  random_state=42, n_jobs=-1)
rf_fold.fit(X[tr], y[tr])
y_pred = rf_fold.predict(X[te])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y[te], y_pred),
    display_labels=['Não-UWBG', 'UWBG']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusão — Random Forest (fold 5)')

# Curva ROC
from sklearn.metrics import roc_curve
y_prob = rf_fold.predict_proba(X[te])[:, 1]
fpr, tpr, _ = roc_curve(y[te], y_prob)
auc_val = roc_auc_score(y[te], y_prob)
axes[1].plot(fpr, tpr, color=C_UWBG, lw=2, label=f'AUC = {auc_val:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=0.8)
axes[1].set_xlabel('Taxa de Falsos Positivos')
axes[1].set_ylabel('Taxa de Verdadeiros Positivos')
axes[1].set_title('Curva ROC — Random Forest')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'classifier_evaluation.png', bbox_inches='tight')
plt.show()


## 8. Importância das Features


In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEAT_COLS)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
colors_imp = [C_UWBG if imp > importances.median() else '#cccccc' for imp in importances]
ax.barh(importances.index, importances.values, color=colors_imp)
ax.set_xlabel('Importância (Mean Decrease in Impurity)')
ax.set_title('Importância das Features — Random Forest')
ax.axvline(importances.median(), color='black', linestyle=':', linewidth=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / 'feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 10 features:')
print(importances.sort_values(ascending=False).head(10).round(4).to_string())


## 9. Filtros Químicos — Exportação para Gen/02

Derivamos regras simples e exportamos o classificador treinado (`S_chem`) para uso
no Experimento Gen/02 como função de score químico.


In [ ]:
# ── Regras derivadas dos dados ──────────────────────────────────────────────
df_uwbg_full = df_feat_full[df_feat_full['is_uwbg'] == 1]
df_nuwbg_full = df_feat_full[df_feat_full['is_uwbg'] == 0]

# Elementos favoráveis: presença em > 10% dos UWBG E razão enriquecimento > 2
fav_elements = df_elem[
    (df_elem['freq_uwbg'] > 10) & (df_elem['ratio'] > 2)
]['element'].tolist()

# Elementos desfavoráveis: razão < 0.5 E presentes em > 5% dos não-UWBG
unfav_elements = df_elem[
    (df_elem['ratio'] < 0.5) & (df_elem['freq_nuwbg'] > 5)
]['element'].tolist()

print(f'Elementos favoráveis UWBG: {sorted(fav_elements)}')
print(f'Elementos desfavoráveis  : {sorted(unfav_elements)}')

# Thresholds de eletronegatividade
mean_en_threshold = df_uwbg_full['mean_en'].quantile(0.25)  # corte conservador
range_en_threshold = df_uwbg_full['range_en'].quantile(0.25)

print(f'\nThreshold mean_en  (Q25 UWBG): {mean_en_threshold:.2f}')
print(f'Threshold range_en (Q25 UWBG): {range_en_threshold:.2f}')

# Salvar filtros como JSON
filters = {
    'uwbg_threshold_eV'       : UWBG_THR,
    'favorable_elements'      : sorted(fav_elements),
    'unfavorable_elements'    : sorted(unfav_elements),
    'mean_en_min'             : round(mean_en_threshold, 3),
    'range_en_min'            : round(range_en_threshold, 3),
    'avoid_transition_metals' : True,
    'feature_cols'            : FEAT_COLS,
    'notes': (
        'favorable_elements: presença em >10% dos UWBG E razão enriquecimento >2. '
        'Usar como boost nos algoritmos de geração (Gen/02).'
    ),
}

with open(OUT_DIR / 'chemical_filters.json', 'w') as f:
    json.dump(filters, f, indent=2)

print('\nFiltros salvos em outputs/chemical_filters.json')

# Salvar classificador
joblib.dump({'model': rf, 'feature_cols': FEAT_COLS, 'scaler': None},
            OUT_DIR / 'rf_classifier.joblib')
joblib.dump({'model': pipe_lr, 'feature_cols': FEAT_COLS},
            OUT_DIR / 'lr_classifier.joblib')
print('Classificadores salvos em outputs/')


In [ ]:
# ── Demonstração: pontuar os candidatos novos (Novel) com S_chem ──────────
df_novel = df_cands_feat[df_cands_feat['class'] == 'Novel'].copy()

X_novel = df_novel[FEAT_COLS].dropna()
if len(X_novel) > 0:
    df_novel = df_novel.loc[X_novel.index].copy()
    df_novel['s_chem_rf'] = rf.predict_proba(X_novel.values)[:, 1]
    df_novel['s_chem_lr'] = pipe_lr.predict_proba(X_novel.values)[:, 1]
    df_novel = df_novel.sort_values('s_chem_rf', ascending=False)
    print(f'Novel candidates pontuados: {len(df_novel)}')
    print('\nTop 15 por S_chem (Random Forest):')
    print(df_novel[['uid', 'formula', 'gap_pbe', 'gap_hse_pred',
                     's_chem_rf', 's_chem_lr']].head(15).to_string(index=False))
    df_novel.to_csv(OUT_DIR / 'novel_candidates_scored.csv', index=False)
else:
    print('Sem candidatos novel com features completas disponíveis.')


## 10. Resumo Executivo


In [ ]:
print('=' * 60)
print('EXPERIMENTO GEN/01 — RESUMO')
print('=' * 60)
print(f'C2DB com HSE calculado     : {len(df_hse):,} materiais')
print(f'  UWBG verdadeiro (>{UWBG_THR} eV) : {df_hse["is_uwbg"].sum():,} ({df_hse["is_uwbg"].mean()*100:.1f}%)')
print()
print('Candidatos run 005 (UWBG predito):')
print(f'  TP (HSE confirmado > 3.4 eV) : {(df_uwbg["class"]=="TP").sum()}')
print(f'  FP (HSE confirmado ≤ 3.4 eV) : {(df_uwbg["class"]=="FP").sum()}')
print(f'  Novel (sem HSE calculado)     : {(df_uwbg["class"]=="Novel").sum()}')
print()
print('Classificador S_chem (5-fold CV):')
print(f'  Logistic Regression — F1: {cv_f1_lr.mean():.3f} | AUC: {cv_auc_lr.mean():.3f}')
print(f'  Random Forest       — F1: {cv_f1_rf.mean():.3f} | AUC: {cv_auc_rf.mean():.3f}')
print()
print('Elementos favoráveis para UWBG:')
print(f'  {sorted(fav_elements)}')
print()
print('Saídas geradas:')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  outputs/{p.name}')
for p in sorted(FIG_DIR.iterdir()):
    print(f'  figures/{p.name}')
